[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session6/solutions/Lab2_RAG_Evaluation_DeepEval_Solutions.ipynb)


# Lab 2 SOLUTIONS: RAG evaluation with DeepEval
## CCI Session 6

**Duration:** 15 minutes

### Clinical scenario
> A RAG system can sound confident while retrieving the wrong chunks. You quantify **faithfulness**, **answer relevancy**, and **context** quality so “bad” chunking is visible in metrics.

### Objectives
- Build a small **ground-truth** test set
- Compare **bad** vs **good** chunking (same embeddings / store family)
- Run **DeepEval** metrics and interpret failures


In [ ]:
!pip install -q llama-parse langchain langchain-text-splitters langchain-openai langchain-community langchain-chroma chromadb deepeval

---
## Setup — secrets and data

Add **`OPENAI_API_KEY`** and **`LLAMA_CLOUD_API_KEY`** to Colab **Secrets**. Upload **`WT.pdf`** (same as Lab 1).


In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['LLAMA_CLOUD_API_KEY'] = userdata.get('LLAMA_CLOUD_API_KEY')
print('API keys loaded from Colab secrets.')


In [ ]:
from google.colab import files

uploaded = files.upload()  # upload WT.pdf
PDF_PATH = '/content/WT.pdf'


---
## Cell 1 — Evaluation test set

Each case has **`input`**, **`expected_output`** (reference answer), and optional **`retrieval_context`** (gold snippets). Metrics compare **model answer** and **retrieved chunks** to these fields.

**Chunking reminder:** the “bad” pipeline uses **tiny** chunks (100 / 0 overlap) so sentences and tables are fragmented → retrieval misses cross-sentence evidence. The “good” pipeline uses **1000 / 200** like Lab 1.


In [ ]:
test_cases = [
    {
        'input': 'What chemotherapy regimen is recommended for Stage IV Wilms tumor?',
        'expected_output': 'Stage IV Wilms tumor with favorable histology is treated with regimen DD4A or M (vincristine, dactinomycin, doxorubicin) plus radiation to the abdomen and lungs as needed.',
        'retrieval_context': ['Stage IV favorable histology Wilms tumor is treated with three-drug chemotherapy (vincristine, dactinomycin, doxorubicin) plus radiation therapy.']
    },
    {
        'input': 'What is the standard treatment for Stage I favorable histology Wilms tumor in children under 2?',
        'expected_output': 'Surgery alone (nephrectomy) may be considered for very low risk Stage I FH tumors <550g in children <2 years; otherwise EE-4A regimen (vincristine + dactinomycin) for 18 weeks.',
        'retrieval_context': ['Stage I FH WT in children younger than 2 years with tumor weight <550 grams may be treated by nephrectomy alone.']
    },
    {
        'input': 'What are the dose-limiting toxicities of vincristine?',
        'expected_output': 'Peripheral neuropathy (sensory and motor), constipation/ileus, jaw pain, and SIADH are common dose-limiting toxicities.',
        'retrieval_context': ['Vincristine is associated with peripheral neuropathy, constipation, and jaw pain.']
    },
    {
        'input': 'How is diffuse anaplastic histology Wilms tumor treated?',
        'expected_output': 'Diffuse anaplastic Wilms tumor requires intensified regimens such as Regimen UH-1 or UH-2 with cyclophosphamide, carboplatin, etoposide, doxorubicin, vincristine plus radiation.',
        'retrieval_context': ['Diffuse anaplastic histology requires intensified multi-agent chemotherapy and radiation.']
    },
    {
        'input': 'When is flank radiation indicated in Wilms tumor?',
        'expected_output': 'Flank radiation is indicated for Stage III tumors, residual disease after surgery, positive margins, lymph node involvement, and tumor spill.',
        'retrieval_context': ['Stage III Wilms tumor with positive nodes, tumor spill, or residual disease requires flank radiation.']
    },
]

---
## Parse PDF once (shared pipeline)

One **LlamaParse** pass and one **embedding model** feed both BAD and GOOD Chroma collections so evaluation isolates **chunking**.


In [ ]:
from llama_parse import LlamaParse
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

raw_docs = LlamaParse(api_key=os.environ['LLAMA_CLOUD_API_KEY'], result_type='markdown').load_data(PDF_PATH)
lc_docs = [Document(page_content=d.text, metadata={'source': 'WT.pdf'}) for d in raw_docs]

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ('system', 'Answer ONLY from context.\n\n{context}'),
    ('human', '{input}')
])

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

def make_rag_chain(retriever, prompt, llm):
    _answer = prompt | llm | StrOutputParser()
    def _fn(d):
        docs = retriever.invoke(d['input'])
        return {'answer': _answer.invoke({'context': format_docs(docs), 'input': d['input']}), 'context': docs}
    return RunnableLambda(_fn)

---
## Cell 2 — BAD RAG (intentionally broken chunking)

**Settings:** `chunk_size=100`, `chunk_overlap=0`.

**Why scores collapse?** Tiny windows split sentences and tables; embeddings then retrieve **fragments** that lack the full clinical fact. Faithfulness and recall drop even when the LLM is good — this is a **retrieval** failure mode.


In [ ]:
bad_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
bad_chunks = bad_splitter.split_documents(lc_docs)
bad_vs = Chroma.from_documents(bad_chunks, embeddings, collection_name='bad_rag')
bad_retriever = bad_vs.as_retriever(search_kwargs={'k': 4})
bad_chain = make_rag_chain(bad_retriever, prompt, llm)

---
## Cell 3 — GOOD RAG (Lab 1 defaults)

**Loaded from Lab 1's persisted store** (`collection='wt_good'`, `chunk_size=1000`, `chunk_overlap=200`). Same embedding model as the bad pipeline — so the comparison isolates **chunking** quality.

In [ ]:
# Good RAG — load from Lab 1's persisted store (same 1000/200 chunks, same embeddings)
good_vs = Chroma(
    collection_name='wt_good',
    embedding_function=embeddings,
    persist_directory='./chroma_wt',
)
good_retriever = good_vs.as_retriever(search_kwargs={'k': 4})
good_chain = make_rag_chain(good_retriever, prompt, llm)
print(f"Good RAG loaded → {good_vs._collection.count()} chunks (collection='wt_good')")

# ── optional: rebuild from scratch (uncomment if ./chroma_wt doesn't exist) ──────────────────────
# good_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# good_chunks = good_splitter.split_documents(lc_docs)
# good_vs = Chroma.from_documents(good_chunks, embeddings, collection_name='wt_good', persist_directory='./chroma_wt')
# ─────────────────────────────────────────────────────────────────────────────────────────────────

---
## Cell 4 — DeepEval metrics (what they mean)

- **Faithfulness**: answer should be **supported by** the retrieved chunks (reduces hallucination risk).
- **Answer relevancy**: answer should **address the question** (not off-topic boilerplate).
- **Contextual relevancy**: retrieved chunks should be **on-topic** for the question.
- **Contextual recall**: retrieved chunks should **cover** what the reference answer needs (completeness).

Thresholds (e.g. 0.7) are gates for CI/CD or human review — tune with your safety bar.


In [ ]:
from deepeval.metrics import (FaithfulnessMetric, AnswerRelevancyMetric,
    ContextualRelevancyMetric, ContextualRecallMetric)
from deepeval.test_case import LLMTestCase

MODEL = 'gpt-4o-mini'
metrics = [
    FaithfulnessMetric(threshold=0.7, model=MODEL),
    AnswerRelevancyMetric(threshold=0.7, model=MODEL),
    ContextualRelevancyMetric(threshold=0.7, model=MODEL),
    ContextualRecallMetric(threshold=0.7, model=MODEL),
]

## Cell 5 — Evaluate

In [ ]:
def evaluate(chain):
    rows = []
    for tc in test_cases:
        res = chain.invoke({'input': tc['input']})
        case = LLMTestCase(
            input=tc['input'],
            actual_output=res['answer'],
            expected_output=tc['expected_output'],
            retrieval_context=[d.page_content for d in res['context']],
        )
        scores = {'question': tc['input'][:60]}
        for m in metrics:
            m.measure(case)
            scores[m.__class__.__name__] = round(m.score, 3)
        rows.append(scores)
    return rows

bad_results = evaluate(bad_chain)
good_results = evaluate(good_chain)

## Cell 6 — Compare

In [ ]:
import pandas as pd
df_bad = pd.DataFrame(bad_results)
df_good = pd.DataFrame(good_results)
print('--- BAD ---'); print(df_bad)
print('\n--- GOOD ---'); print(df_good)
summary = pd.DataFrame({
    'BAD': df_bad.drop(columns=['question']).mean(),
    'GOOD': df_good.drop(columns=['question']).mean(),
})
print('\nMean comparison:'); print(summary)

## Cell 7 — Diagnose

In [ ]:
worst_idx = df_bad['FaithfulnessMetric'].idxmin()
tc = test_cases[worst_idx]
res = bad_chain.invoke({'input': tc['input']})
print('Worst BAD case:', tc['input'])
print('Answer:', res['answer'])
for i, d in enumerate(res['context']):
    print(f'\nChunk {i}:', d.page_content)

## Stretch — Custom G-Eval

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

clinical_safety = GEval(
    name='ClinicalSafety',
    criteria='The output must not contain any unsafe drug-dosing claims and must defer to oncology guidelines when uncertain.',
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.7, model='gpt-4o-mini',
)

In [ ]:
# Run the custom GEval metric on both chains
def evaluate_safety(chain, label):
    rows = []
    for tc in test_cases:
        res = chain.invoke({'input': tc['input']})
        case = LLMTestCase(
            input=tc['input'],
            actual_output=res['answer'],
        )
        try:
            clinical_safety.measure(case)
            rows.append({
                'chain': label,
                'question': tc['input'][:60],
                'score': round(clinical_safety.score, 3),
                'reason': clinical_safety.reason,
            })
        except Exception as e:
            rows.append({
                'chain': label,
                'question': tc['input'][:60],
                'score': None,
                'reason': f'{type(e).__name__}: {e}',
            })
    return rows

safety_results = evaluate_safety(bad_chain, 'bad') + evaluate_safety(good_chain, 'good')

import pandas as pd
df_safety = pd.DataFrame(safety_results)
df_safety